**BANK CUSTOMER CHURN PREDICTION**
###### - ANANYA PATANKAR


---



The LightGBM model performed best during model evaluation and hence is used for hyperparameter tuning.

Hyperparameter tuning helps improve a model by finding the combination of parameters that leads to better predictive performance.

Optuna is used because it can efficiently search through different parameter combinations and identify those that produce the best results.

F1 is chosen as optimization metric as it balances both precision and recall.

In [1]:
!pip install optuna -q

In [2]:
#import required libraries
import optuna
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
#load processed datasets
X_train = pd.read_csv('X_train_final.csv')
X_test = pd.read_csv('X_test_final.csv')
y_train = pd.read_csv('y_train_final.csv').squeeze()
y_test = pd.read_csv('y_test_final.csv').squeeze()

**HYPERPARAMETER TUNING :**

In [4]:
#define search space and objective function for Optuna to optimize
def objective(trial):
  params = {
      'n_estimators': trial.suggest_int('n_estimators', 100, 500),
      'max_depth': trial.suggest_int('max_depth', 3, 12),
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
      'num_leaves': trial.suggest_int('num_leaves', 20, 150),
      'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
      'subsample': trial.suggest_float('subsample', 0.5, 1.0),
      'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
      'random_state': 42,
      'verbose': -1
    }
  model = LGBMClassifier(**params)
  score = cross_val_score(model, X_train, y_train, cv=5, scoring='f1').mean()
  return score

In [5]:
#run optimization across 50 trials
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=50)
print("Best trial number:", study.best_trial.number)
print("Best F1 (CV):", study.best_value)
print("Best params:", study.best_params)

[I 2026-06-21 15:10:53,877] A new study created in memory with name: no-name-99783f41-45c7-425d-b86d-4784fb3a55c9
[I 2026-06-21 15:11:07,201] Trial 0 finished with value: 0.8748978302372384 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'num_leaves': 98, 'min_child_samples': 19, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998}. Best is trial 0 with value: 0.8748978302372384.
[I 2026-06-21 15:11:19,411] Trial 1 finished with value: 0.867032914498496 and parameters: {'n_estimators': 447, 'max_depth': 9, 'learning_rate': 0.11114989443094977, 'num_leaves': 22, 'min_child_samples': 98, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381}. Best is trial 0 with value: 0.8748978302372384.
[I 2026-06-21 15:11:23,013] Trial 2 finished with value: 0.8408282042837945 and parameters: {'n_estimators': 172, 'max_depth': 4, 'learning_rate': 0.028145092716060652, 'num_leaves': 88, 'min_child_samples': 46, 'subsampl

Best trial number: 28
Best F1 (CV): 0.8809886005855916
Best params: {'n_estimators': 397, 'max_depth': 9, 'learning_rate': 0.03681608882483862, 'num_leaves': 87, 'min_child_samples': 17, 'subsample': 0.6768746712684484, 'colsample_bytree': 0.6385065337392571}


In [6]:
#train final model using best parameters found
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = -1

lgbm_tuned = LGBMClassifier(**best_params)
lgbm_tuned.fit(X_train, y_train)

LGBMClassifier(colsample_bytree=0.6385065337392571,
               learning_rate=0.03681608882483862, max_depth=9,
               min_child_samples=17, n_estimators=397, num_leaves=87,
               random_state=42, subsample=0.6768746712684484, verbose=-1)

Optuna ran 50 trials, with the best result found at Trial 28. The best parameters found were: 397 trees, max depth of 9, learning rate of 0.037, and 87 leaves.



**EVALUATING ON TEST DATA :**

In [7]:
#evaluate tuned model on test set
y_pred_tuned = lgbm_tuned.predict(X_test)
y_pred_proba_tuned = lgbm_tuned.predict_proba(X_test)[:,1]
print("Tuned LightGBM:")
print(f"Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_tuned):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_tuned):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_tuned):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")

Tuned LightGBM:
Accuracy : 0.8570
Precision: 0.6971
Recall   : 0.5258
F1 Score : 0.5994
ROC-AUC  : 0.8478


**COMPARING WITH UNTUNED MODEL :**

In [8]:
#load untuned model
lgbm_untuned = joblib.load('lightgbm.pkl')
y_pred_untuned = lgbm_untuned.predict(X_test)
y_pred_proba_untuned = lgbm_untuned.predict_proba(X_test)[:,1]

In [9]:
#compare tuned and untuned models
comparison = pd.DataFrame({'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
                           'Untuned': [accuracy_score(y_test, y_pred_untuned),
                                       precision_score(y_test, y_pred_untuned),
                                       recall_score(y_test, y_pred_untuned),
                                       f1_score(y_test, y_pred_untuned),
                                       roc_auc_score(y_test, y_pred_proba_untuned)],
                           'Tuned': [accuracy_score(y_test, y_pred_tuned),
                                     precision_score(y_test, y_pred_tuned),
                                     recall_score(y_test, y_pred_tuned),
                                     f1_score(y_test, y_pred_tuned),
                                     roc_auc_score(y_test, y_pred_proba_tuned)]
                           })

comparison['Untuned'] = comparison['Untuned'].map('{:.2%}'.format)
comparison['Tuned'] = comparison['Tuned'].map('{:.2%}'.format)
display(comparison)

,Metric,Untuned,Tuned
0,Accuracy,86.10%,85.70%
1,Precision,69.25%,69.71%
2,Recall,57.00%,52.58%
3,F1 Score,62.53%,59.94%
4,ROC-AUC,85.88%,84.78%


The tuned model shows drops across most metrics compared to the untuned baseline model. Precision improved slightly from 69.25% to 69.71%, but Recall, F1, and ROC-AUC declined.

This suggests the default LightGBM model was already near-optimal for this dataset, and that further performance gains are more likely to come from stronger feature engineering than parameter tuning.

**SAVING THE TUNED MODEL :**

In [10]:
#save tuned model
joblib.dump(lgbm_tuned, 'lightgbm_tuned.pkl')

['lightgbm_tuned.pkl']

The untuned model is carried forward for the final pipeline.